In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)
from rich import print as rprint

# 以init_chat_model为例
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)



agent = create_agent(
    model=model,
    name="agent01"
)

# 调用
response= agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100 + 20 * 3 = ？"}
    ]
})


# rprint(response)

for message in response["messages"]:
    message.pretty_print()

================================ System Message ================================

你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题
================================ Human Message =================================

100 + 20 * 3 = ？
================================== Ai Message ==================================
Name: agent01

先算乘法：20 × 3 = 60  
再算加法：100 + 60 = 160  

所以答案是：**160**。


## 2、系统提示词的设置

使用system_prompt参数进行设置，可以是str,也可以是SystemMessage

举例1：

In [ ]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)


# 导入模型
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 2.导入工具
web_search = TavilySearch(max_results=2)


#系统提示词可以在创建agent的时候配置，也可以在agent调用invoke()方法的时候配置。
#在创建agent时配置的系统提示词不会作为SystemMessage传入
# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)

rprint(result)

举例2：

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"


# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers],  # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "10加上20再加上30是多少？"}
    ]},
)

rprint(response)
# print(response["messages"][-1].content)

## 3、结构化输出的4种策略

### 3.1 ProviderStrategy策略：

In [3]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking":{"type":"disabled"}} ##此处关闭思考模型
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")


#deepseek不支持response_format()方法，需要关闭思考模式
agent = create_agent(
    model = model,
    response_format=ProviderStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
    ]
})

rprint(response)

BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

### 3.2 ToolStrategy策略：

In [4]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking":{"type":"disabled"}}
)


# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ToolStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

# 从返回结果中可以看到，ToolCalls不为空，表明这部分内容被包装为工具进行调用，大致原理是通过包装为虚拟工具规避了一些兼容性的问题
rprint(response)

# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='e746f1d4-9c5b-4a1f-9d9f-0293ca5f9b67'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 348,
                    'total_tokens': 426,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 92
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'ac260513-7c25-4144-b529-5593295d70a3',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02237-ce74-71b3-9aeb-5f2b35654ca0-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_00_TU4FPGojxjVkr8K292PH4425',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 348,
                'output_tokens': 78,
                'total_tokens': 426,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='13012341234'",
            name='ContractInfo',
            id='1a920f69-1c0f-4be6-b0fc-b2f7fc537f97',
            tool_call_id='call_00_TU4FPGojxjVkr8K292PH4425'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

### 3.3 AutoStrategy / Type策略：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking":{"type":"disabled"}}
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=AutoStrategy(ContractInfo)  ##AutoStrategy是会在底层判断使用ProviderStrategy还是使用ToolStrategy
    # response_format=ContractInfo  # 不推荐大家使用
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)

# print(response["structured_response"])